# Whisper vs faster-whisper vs Vosk Multi-File Benchmark

**Instructions:**
1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Run all cells (`Ctrl+F9` or `Cmd+F9`).

This notebook generates 5 distinct audio samples of varying lengths (e.g., ~5s, 15s, 30s, 45s, 60s), installs the transcription libraries, and runs a comprehensive loop evaluating both **speed** and **Word Error Rate (WER)** across all lengths.

In [ ]:
# 1. Install dependencies and download Vosk model
!pip install -q openai-whisper faster-whisper vosk gtts pydub jiwer
!wget -q -nc https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip
!unzip -q -o vosk-model-small-en-us-0.15.zip
print("Dependencies installed and Vosk model downloaded.")

In [42]:
# 2. Generate 5 test audio files of varying lengths
import os
from gtts import gTTS
from pydub import AudioSegment
import re

# Define 5 different texts ranging from ~5s to ~60s
texts = [
    # Test 1: ~5 seconds
    ("The quick brown fox jumps over the lazy dog near the riverbank. This is a short test sentence for the transcription engine."),
    
    # Test 2: ~15 seconds
    ("Artificial intelligence is transforming the way we work and live. By automating repetitive tasks, "
     "machines allow humans to focus on creative problem solving and strategic thinking. Voice assistants "
     "are now embedded in our phones, cars, and homes."),
     
    # Test 3: ~30 seconds
    ("The development of artificial intelligence has been one of the most significant "
     "technological achievements of the twenty first century. Machine learning algorithms "
     "can now process vast amounts of data and identify patterns that would be impossible "
     "for humans to detect. Natural language processing has enabled computers to understand "
     "and generate human language with remarkable accuracy. Speech recognition systems "
     "like Whisper have made it possible to transcribe audio in real time. "),
     
    # Test 4: ~45 seconds
    ("Space exploration has always fascinated humankind, pushing the boundaries of what is possible. "
     "From the early days of the Apollo missions to the modern rovers exploring the surface of Mars, "
     "we continually strive to understand the universe around us. Telescopes like Hubble and James Webb "
     "have captured breathtaking images of distant galaxies and nebulae, revealing a cosmos far more "
     "complex and beautiful than anyone could have imagined. As private companies join national space agencies "
     "in the pursuit of commercial spaceflight and lunar habitats, the next decade promises unprecedented "
     "advancements in our journey to become an interplanetary species, unlocking the secrets of the solar system."),
     
    # Test 5: ~60+ seconds
    ("Climate change remains one of the greatest global challenges of our time, requiring immediate "
     "and coordinated action from governments, corporations, and individuals alike. The scientific consensus "
     "is clear that the Earth's average temperature is rising due to greenhouse gas emissions from human activities "
     "such as burning fossil fuels and deforestation. This warming trend has profound impacts on weather patterns, "
     "leading to more frequent and severe heatwaves, droughts, and powerful storms. Melting polar ice caps "
     "and thermal expansion are causing sea levels to rise, threatening coastal communities worldwide. "
     "Transitioning to renewable energy sources like solar, wind, and hydroelectric power is crucial to mitigate "
     "these effects. Furthermore, adopting sustainable agricultural practices and investing in green technologies "
     "will help build a resilient future for generations to come. The window to act is narrowing, making innovation "
     "and international cooperation more vital than ever.")
]

test_files = []
print("Generating audio clips...")

for idx, raw_text in enumerate(texts):
    # Clean text for strict WER comparison
    ground_truth = re.sub(r'[^\w\s]', '', raw_text).lower()
    
    mp3_path = f"test_audio_{idx+1}.mp3"
    wav_path = f"test_audio_{idx+1}.wav"
    
    tts = gTTS(text=raw_text, lang="en")
    tts.save(mp3_path)
    
    # Convert to 16kHz WAV
    audio = AudioSegment.from_mp3(mp3_path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    duration_secs = len(audio) / 1000.0
    audio.export(wav_path, format="wav")
    
    test_files.append({
        'id': idx + 1,
        'wav_path': wav_path,
        'duration': duration_secs,
        'ground_truth': ground_truth
    })
    print(f"Generated Test {idx+1}: {duration_secs:.1f} seconds")


Generating audio clips...
Generated Test 1: 8.7 seconds
Generated Test 2: 19.9 seconds
Generated Test 3: 34.8 seconds
Generated Test 4: 51.7 seconds
Generated Test 5: 74.5 seconds


In [43]:
# 3. Benchmark OpenAI Whisper
import whisper
import time
import torch
from jiwer import wer
import re

model_name = "small"
print(f"\n========== OPENAI WHISPER (Local GPU, {model_name}) ==========")

t0 = time.time()
model_openai = whisper.load_model(model_name)
load_time_ow = time.time() - t0
print(f"Model load time: {load_time_ow:.2f}s")

results_ow = []

for test in test_files:
    print(f"\n--- Running Test {test['id']} ({test['duration']:.1f}s) ---")
    
    # Warmup
    model_openai.transcribe(test['wav_path'], language="en", temperature=0)
    
    times = []
    final_text = ""
    for i in range(3):
        t0 = time.time()
        res = model_openai.transcribe(test['wav_path'], language="en", temperature=0)
        times.append(time.time() - t0)
        final_text = res['text']
        
    avg_time = sum(times) / len(times)
    clean_text = re.sub(r'[^\w\s]', '', final_text).lower().strip()
    current_wer = wer(test['ground_truth'], clean_text)
    
    results_ow.append({'id': test['id'], 'time': avg_time, 'wer': current_wer})
    print(f"  Transcribe Time: {avg_time:.2f}s | WER: {current_wer:.2%}")

del model_openai
torch.cuda.empty_cache()


========== OPENAI WHISPER (Local GPU, small) ==========
Model load time: 4.71s

--- Running Test 1 (8.7s) ---
  Transcribe Time: 0.63s | WER: 0.00%

--- Running Test 2 (19.9s) ---
  Transcribe Time: 1.00s | WER: 0.00%

--- Running Test 3 (34.8s) ---
  Transcribe Time: 1.85s | WER: 4.29%

--- Running Test 4 (51.7s) ---
  Transcribe Time: 2.77s | WER: 0.00%

--- Running Test 5 (74.5s) ---
  Transcribe Time: 3.51s | WER: 0.00%


In [44]:
# 4. Benchmark faster-whisper
from faster_whisper import WhisperModel
import time
import torch
from jiwer import wer
import re

compute_type = "float16"
print(f"\n========== FASTER-WHISPER (Local GPU, {model_name}, {compute_type}) ==========")

t0 = time.time()
model_faster = WhisperModel(model_name, device="cuda", compute_type=compute_type)
load_time_fw = time.time() - t0
print(f"Model load time: {load_time_fw:.2f}s")

results_fw = []

for test in test_files:
    print(f"\n--- Running Test {test['id']} ({test['duration']:.1f}s) ---")
    
    # Warmup
    list(model_faster.transcribe(test['wav_path'], language="en", temperature=0)[0])
    
    times = []
    final_text = ""
    for i in range(3):
        t0 = time.time()
        segments, info = model_faster.transcribe(test['wav_path'], language="en", temperature=0)
        final_text = " ".join([s.text for s in segments])
        times.append(time.time() - t0)
        
    avg_time = sum(times) / len(times)
    clean_text = re.sub(r'[^\w\s]', '', final_text).lower().strip()
    current_wer = wer(test['ground_truth'], clean_text)
    
    results_fw.append({'id': test['id'], 'time': avg_time, 'wer': current_wer})
    print(f"  Transcribe Time: {avg_time:.2f}s | WER: {current_wer:.2%}")

del model_faster
torch.cuda.empty_cache()


========== FASTER-WHISPER (Local GPU, small, float16) ==========
Model load time: 0.69s

--- Running Test 1 (8.7s) ---
  Transcribe Time: 0.38s | WER: 0.00%

--- Running Test 2 (19.9s) ---
  Transcribe Time: 0.51s | WER: 0.00%

--- Running Test 3 (34.8s) ---
  Transcribe Time: 0.88s | WER: 4.29%

--- Running Test 4 (51.7s) ---
  Transcribe Time: 1.27s | WER: 0.00%

--- Running Test 5 (74.5s) ---
  Transcribe Time: 1.46s | WER: 0.00%


In [45]:
# 5. Benchmark Vosk (Offline/CPU)
from vosk import Model, KaldiRecognizer
import time
import wave
import json
from jiwer import wer
import re

print("\n========== VOSK (CPU-Only, Offline) ==========")

t0 = time.time()
model_vosk = Model("vosk-model-small-en-us-0.15")
load_time_vosk = time.time() - t0
print(f"Model load time: {load_time_vosk:.2f}s")

results_vosk = []

for test in test_files:
    print(f"\n--- Running Test {test['id']} ({test['duration']:.1f}s) ---")
    
    times = []
    final_text = ""
    
    for i in range(3):
        t0 = time.time()
        wf = wave.open(test['wav_path'], "rb")
        rec = KaldiRecognizer(model_vosk, wf.getframerate())
        rec.SetWords(True)
        
        transcript_chunks = []
        while True:
            data = wf.readframes(4000)
            if len(data) == 0:
                break
            if rec.AcceptWaveform(data):
                res = json.loads(rec.Result())
                transcript_chunks.append(res.get("text", ""))

        final_res = json.loads(rec.FinalResult())
        transcript_chunks.append(final_res.get("text", ""))
        
        final_text = " ".join(transcript_chunks).strip()
        times.append(time.time() - t0)
        wf.close()
        
    avg_time = sum(times) / len(times)
    clean_text = re.sub(r'[^\w\s]', '', final_text).lower().strip()
    current_wer = wer(test['ground_truth'], clean_text)
    
    results_vosk.append({'id': test['id'], 'time': avg_time, 'wer': current_wer})
    print(f"  Transcribe Time: {avg_time:.2f}s | WER: {current_wer:.2%}")

del model_vosk


========== VOSK (CPU-Only, Offline) ==========
Model load time: 0.62s

--- Running Test 1 (8.7s) ---
  Transcribe Time: 1.76s | WER: 0.00%

--- Running Test 2 (19.9s) ---
  Transcribe Time: 2.72s | WER: 2.70%

--- Running Test 3 (34.8s) ---
  Transcribe Time: 4.00s | WER: 1.43%

--- Running Test 4 (51.7s) ---
  Transcribe Time: 5.03s | WER: 4.81%

--- Running Test 5 (74.5s) ---
  Transcribe Time: 6.36s | WER: 5.76%


In [46]:
# 6. Print Comprehensive Multi-File Results
import pandas as pd

print("\n================ OVERALL SUMMARY ================")
print("Hardware: Tesla T4 GPU (Google Colab)\n")

print("--- Load Times ---")
print(f"OpenAI Whisper: {load_time_ow:.2f}s")
print(f"faster-whisper: {load_time_fw:.2f}s  (loaded {load_time_ow/load_time_fw:.1f}x faster)")
print(f"Vosk (CPU):     {load_time_vosk:.2f}s\n")

print("--- Transcription Time by Audio Length ---")
speed_data = {
    'Test ID': [t['id'] for t in test_files],
    'Audio Length (s)': [f"{t['duration']:.1f}s" for t in test_files],
    'OpenAI Whisper (GPU)': [f"{r['time']:.2f}s" for r in results_ow],
    'faster-whisper (GPU)': [f"{r['time']:.2f}s" for r in results_fw],
    'Vosk (CPU)': [f"{r['time']:.2f}s" for r in results_vosk],
    'FW Speedup vs Whisper': [f"{ow['time'] / fw['time']:.1f}x" for ow, fw in zip(results_ow, results_fw)]
}
df_speed = pd.DataFrame(speed_data)
print(df_speed.to_string(index=False))
print("\n")

print("--- Average Word Error Rate (WER) ---")
wer_data = {
    'Test ID': [t['id'] for t in test_files],
    'OpenAI Whisper': [f"{r['wer']:.2%}" for r in results_ow],
    'faster-whisper': [f"{r['wer']:.2%}" for r in results_fw],
    'Vosk': [f"{r['wer']:.2%}" for r in results_vosk]
}
df_wer = pd.DataFrame(wer_data)
print(df_wer.to_string(index=False))

avg_speedup = sum([ow['time'] / fw['time'] for ow, fw in zip(results_ow, results_fw)]) / len(results_ow)
print(f"\n🔥 ON AVERAGE, FASTER-WHISPER IS {avg_speedup:.1f}x FASTER THAN STANDARD WHISPER ACROSS ALL CLIP LENGTHS! 🔥")


================ OVERALL SUMMARY ================
Hardware: Tesla T4 GPU (Google Colab)

--- Load Times ---
OpenAI Whisper: 4.71s
faster-whisper: 0.69s  (loaded 6.8x faster)
Vosk (CPU):     0.62s

--- Transcription Time by Audio Length ---
 Test ID Audio Length (s) OpenAI Whisper (GPU) faster-whisper (GPU) Vosk (CPU) FW Speedup vs Whisper
       1             8.7s                0.63s                0.38s      1.76s                  1.6x
       2            19.9s                1.00s                0.51s      2.72s                  2.0x
       3            34.8s                1.85s                0.88s      4.00s                  2.1x
       4            51.7s                2.77s                1.27s      5.03s                  2.2x
       5            74.5s                3.51s                1.46s      6.36s                  2.4x


--- Average Word Error Rate (WER) ---
 Test ID OpenAI Whisper faster-whisper  Vosk
       1          0.00%          0.00% 0.00%
       2          0.00%